In [16]:
import os

from dotenv import load_dotenv

_ = load_dotenv()

In [10]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun

In [13]:
tool = DuckDuckGoSearchRun()

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [8]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [14]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [17]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY", "").strip()
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").strip().rstrip("/")
deepseek_model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat").strip()

if not deepseek_api_key:
    raise RuntimeError("DEEPSEEK_API_KEY is missing from .env")

if not deepseek_base_url.endswith("/v1"):
    deepseek_base_url = f"{deepseek_base_url}/v1"

model = ChatOpenAI(
    model=deepseek_model,
    api_key=deepseek_api_key,
    base_url=deepseek_base_url,
)
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [18]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [19]:
thread = {"configurable": {"thread_id": "1"}}

In [20]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content="I'll search for the current weather in San Francisco.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 405, 'total_tokens': 465, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 405}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '72f998c8-1994-43f7-8370-ae94eee0a5a3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d170f-af33-70c2-acef-4923ef658ef3-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'current weather San Francisco'}, 'id': 'call_00_edt4XeZt2c8h4ltiVsGtCSgE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 405, 'output_tokens': 60, 'total_tokens': 465, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})]
Cal

In [21]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="I'll search for the current weather in Los Angeles.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 1214, 'total_tokens': 1274, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 1152}, 'prompt_cache_hit_tokens': 1152, 'prompt_cache_miss_tokens': 62}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '2c018ffa-822d-4825-b383-8361fed67863', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d171d-6602-7aa0-9769-42bd042c54b4-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'current weather Los Angeles'}, 'id': 'call_00_a9jsBVoxyiRUh7t3A9Jngc9u', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1214, 'output_tokens': 60, 'total_tokens': 1274, 'input_token_details': {'cache_read': 1152}, 'output_toke

In [22]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Based on the information I found from the searches:\n\n**San Francisco**: 8°C (46°F) currently, with a high of 16°C (61°F)\n\n**Los Angeles**: Around 13°C (55°F) currently (though one source mentioned 27°C/81°F, which seems inconsistent)\n\nComparing the most consistent data:\n- San Francisco: 8°C (46°F)\n- Los Angeles: 13°C (55°F)\n\n**Los Angeles appears to be warmer** than San Francisco right now. The temperature difference is about 5°C (9°F), with Los Angeles being noticeably warmer.\n\nThis makes sense geographically and climatically - Los Angeles is further south and typically has warmer temperatures than San Francisco, especially with San Francisco's famous fog and cooler coastal influence.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 172, 'prompt_tokens': 2138, 'total_tokens': 2310, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2112}, 'prompt_

In [23]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="I need more information to answer your question about which one is warmer. Could you please specify what you're comparing? For example:\n\n- Are you comparing two locations (like cities or countries)?\n- Are you comparing two materials or substances?\n- Are you comparing two time periods?\n- Or something else entirely?\n\nOnce you tell me what you're comparing, I can help you determine which one is warmer.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 403, 'total_tokens': 483, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 19}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache_new_kvcache', 'id': '6a15724f-4dcb-4374-b321-402d84e5244a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d171e-ec